# Season orchestrator quickstart

This notebook shows a minimal, self-contained example of wiring up the
`SeasonOrchestrator` without relying on the full simulation stack. To keep
the example runnable in lightweight environments, we stub out the
heavy dependencies (`geopandas`, `TrafficModel`, and the analysis
utilities) and then execute a short two-day season run.

In [ ]:
import sys
import types
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

# --- Lightweight stubs for external dependencies ---
# geopandas stub: just return a small DataFrame when read_parquet is called
geopandas_stub = types.ModuleType("geopandas")
geopandas_stub.read_parquet = lambda path: pd.DataFrame({"road_id": [1, 2], "length_m": [100, 250]})
sys.modules["geopandas"] = geopandas_stub

# traffic.utils.analysis_utils stub: produce simple DataFrames expected by the orchestrator
analysis_utils_stub = types.ModuleType("traffic.utils.analysis_utils")

def _simple_vehicle_ts(tm, plots=False):
    return pd.DataFrame({"day": [tm.day_index], "avg_speed": [tm.traffic_percentile or 0]})

def _simple_model_ts(tm):
    return pd.DataFrame({"day": [tm.day_index], "steps": [tm.schedule.steps]})

def _finished_agents(tm, plots=False):
    return pd.DataFrame({"day": [tm.day_index], "finished": [tm.max_persons]})

analysis_utils_stub.vehicle_agent_data_time_series = _simple_vehicle_ts
analysis_utils_stub.model_data_time_series = _simple_model_ts
analysis_utils_stub.finished_agents_summary_df = _finished_agents

traffic_utils_stub = types.ModuleType("traffic.utils")
traffic_utils_stub.analysis_utils = analysis_utils_stub
sys.modules["traffic.utils"] = traffic_utils_stub
sys.modules["traffic.utils.analysis_utils"] = analysis_utils_stub

# traffic.model.traffic_model stub: provides a minimal TrafficModel class
traffic_model_module = types.ModuleType("traffic.model.traffic_model")

class TrafficModel:
    def __init__(self, *, road_gdf, ecs_df, season_persons, max_steps, seed, batchrun, collect_every_n, start_hr, traffic_percentile, p_generate, max_persons, canyon_closures, bus_interval, car_preference, bus_capacity, crashes_per_100k_vmt_input):
        self.road_gdf = road_gdf
        self.ecs_df = ecs_df
        self.season_persons = season_persons
        self.max_steps = max_steps
        self.seed = seed
        self.traffic_percentile = traffic_percentile
        self.bus_interval = bus_interval
        self.max_persons = max_persons
        self.crashes_per_100k_vmt_input = crashes_per_100k_vmt_input
        self.day_index = canyon_closures.get("day_index", 0)
        self.schedule = types.SimpleNamespace(steps=0)

    def run_model(self):
        # Pretend the model advanced a few steps
        self.schedule.steps = self.max_steps // 10

traffic_model_module.TrafficModel = TrafficModel

traffic_model_package = types.ModuleType("traffic.model")
traffic_model_package.traffic_model = traffic_model_module
sys.modules["traffic.model"] = traffic_model_package
sys.modules["traffic.model.traffic_model"] = traffic_model_module

traffic_pkg = types.ModuleType("traffic")
traffic_pkg.model = traffic_model_package
traffic_pkg.utils = traffic_utils_stub
sys.modules["traffic"] = traffic_pkg

# With the stubs in place we can import the orchestrator
from season.season_orchestrator import SeasonOrchestrator

In [ ]:
# Define a tiny season configuration that satisfies SeasonOrchestrator's protocol
@dataclass
class SimpleDay:
    day_index: int
    traffic_percentile: int | None
    bus_interval: int
    canyon_closures: dict | None
    crashes_per_100k_vmt_input: float | None

@dataclass
class SimpleSeasonConfig:
    season_id: str
    seed: int
    max_steps: int
    start_hr: int
    max_persons: int
    bus_capacity: int
    road_gdf_path: str | Path
    ecs_df_path: str | Path
    day_params: list[SimpleDay]

    def validate(self) -> None:
        print(f"Validated season '{self.season_id}' with {len(self.day_params)} days")

# Create tiny parquet files that the orchestrator expects to load
road_path = Path("demo_road.parquet")
ecs_path = Path("demo_ecs.parquet")
pd.DataFrame({"road_id": [1], "length_m": [100]}).to_parquet(road_path)
pd.DataFrame({"segment": ["A"], "expected_count": [42]}).to_parquet(ecs_path)

season_config = SimpleSeasonConfig(
    season_id="demo-season",
    seed=7,
    max_steps=100,
    start_hr=6,
    max_persons=5,
    bus_capacity=12,
    road_gdf_path=road_path,
    ecs_df_path=ecs_path,
    day_params=[
        SimpleDay(day_index=0, traffic_percentile=25, bus_interval=15, canyon_closures={"day_index": 0}, crashes_per_100k_vmt_input=3.5),
        SimpleDay(day_index=1, traffic_percentile=50, bus_interval=20, canyon_closures={"day_index": 1}, crashes_per_100k_vmt_input=4.0),
    ],
)

orchestrator = SeasonOrchestrator(season_config, output_dir="demo_outputs")
result = orchestrator.run()
result.day_file_log